# RAG with Azure AI Search + Azure OpenAI — a step-by-step lab

**Retrieval-Augmented Generation (RAG)** lets a language model answer questions about *your own documents*.
Instead of relying on what the model memorised during training, we:

1. **Retrieve** the most relevant pieces (chunks) of our documents from a search index,
2. **Augment** the prompt with those chunks, and
3. **Generate** an answer with the LLM — grounded in, and citing, those chunks.

**How to use this notebook:** run the cells **one at a time, top to bottom** (Shift + Enter). Each step builds on the previous one.

| Step | What you learn |
|---|---|
| 0 | The architecture (diagram) |
| 1 | Install packages |
| 2 | Imports & loading secrets from `.env` |
| 3 | Configuration |
| 4 | Authenticating to Azure |
| 5 | Connecting to Azure AI Search |
| 6 | Inspecting the index |
| 7 | **Retrieve** — searching for chunks |
| 8 | **Augment** — building the prompt |
| 9 | Connecting to the LLM |
| 10 | **Generate** — getting the answer |
| 11 | Putting it all together: `ask()` |
| 12 | Try it yourself + exercises |
| 13 | Clean up |

## Step 0 — The architecture

There are two separate flows:

* **A. Indexing (offline):** documents are split into small *chunks*, optionally turned into vectors (embeddings), and stored in an Azure AI Search index. This was already done with the *Import and vectorize data* wizard in the Azure portal.
* **B. Question answering (online):** for every question we run **Retrieve → Augment → Generate**. This notebook focuses on flow B.

In [ ]:
from IPython.display import SVG, display

RAG_DIAGRAM = r'''<svg xmlns="http://www.w3.org/2000/svg" width="900" height="470" viewBox="0 0 900 470" font-family="Segoe UI, Helvetica, Arial, sans-serif">
  <defs>
    <marker id="a" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#475569"/></marker>
  </defs>
  <rect width="900" height="470" fill="#ffffff"/>
  <text x="450" y="30" text-anchor="middle" font-size="20" font-weight="700" fill="#0f172a">Retrieval-Augmented Generation (RAG) on Azure</text>

  <!-- Indexing lane -->
  <rect x="20" y="50" width="860" height="120" rx="10" fill="#f1f5f9" stroke="#cbd5e1"/>
  <text x="35" y="72" font-size="13" font-weight="700" fill="#64748b">A. INDEXING (done once, offline)</text>
  <rect x="50" y="90" width="170" height="60" rx="8" fill="#e2e8f0" stroke="#64748b"/>
  <text x="135" y="116" text-anchor="middle" font-size="14" font-weight="600">Documents</text>
  <text x="135" y="134" text-anchor="middle" font-size="11" fill="#475569">PDF / Word / Blob Storage</text>
  <rect x="290" y="90" width="230" height="60" rx="8" fill="#fef3c7" stroke="#d97706"/>
  <text x="405" y="116" text-anchor="middle" font-size="14" font-weight="600">Indexer + Skillset</text>
  <text x="405" y="134" text-anchor="middle" font-size="11" fill="#475569">extract text → split into chunks → embed</text>
  <rect x="590" y="90" width="250" height="60" rx="8" fill="#dbeafe" stroke="#2563eb"/>
  <text x="715" y="116" text-anchor="middle" font-size="14" font-weight="600">Azure AI Search Index</text>
  <text x="715" y="134" text-anchor="middle" font-size="11" fill="#475569">chunk_id · chunk · title · vector</text>
  <line x1="220" y1="120" x2="288" y2="120" stroke="#475569" stroke-width="2" marker-end="url(#a)"/>
  <line x1="520" y1="120" x2="588" y2="120" stroke="#475569" stroke-width="2" marker-end="url(#a)"/>

  <!-- Query lane -->
  <rect x="20" y="190" width="860" height="265" rx="10" fill="#f8fafc" stroke="#cbd5e1"/>
  <text x="35" y="212" font-size="13" font-weight="700" fill="#64748b">B. QUESTION ANSWERING (every request)</text>

  <rect x="50" y="240" width="170" height="60" rx="8" fill="#ede9fe" stroke="#7c3aed"/>
  <text x="135" y="266" text-anchor="middle" font-size="14" font-weight="600">① User question</text>
  <text x="135" y="284" text-anchor="middle" font-size="11" fill="#475569">"What is earned leave?"</text>

  <rect x="290" y="240" width="230" height="60" rx="8" fill="#dbeafe" stroke="#2563eb"/>
  <text x="405" y="266" text-anchor="middle" font-size="14" font-weight="600">② RETRIEVE</text>
  <text x="405" y="284" text-anchor="middle" font-size="11" fill="#475569">search the index → top-k chunks</text>

  <rect x="590" y="240" width="250" height="60" rx="8" fill="#dcfce7" stroke="#16a34a"/>
  <text x="715" y="266" text-anchor="middle" font-size="14" font-weight="600">③ AUGMENT</text>
  <text x="715" y="284" text-anchor="middle" font-size="11" fill="#475569">prompt = instructions + chunks + question</text>

  <rect x="590" y="360" width="250" height="60" rx="8" fill="#fce7f3" stroke="#db2777"/>
  <text x="715" y="386" text-anchor="middle" font-size="14" font-weight="600">④ GENERATE</text>
  <text x="715" y="404" text-anchor="middle" font-size="11" fill="#475569">Azure OpenAI / Foundry model (LLM)</text>

  <rect x="290" y="360" width="230" height="60" rx="8" fill="#ede9fe" stroke="#7c3aed"/>
  <text x="405" y="386" text-anchor="middle" font-size="14" font-weight="600">⑤ Answer + citations</text>
  <text x="405" y="404" text-anchor="middle" font-size="11" fill="#475569">"Earned leave is ... [1]"</text>

  <line x1="220" y1="270" x2="288" y2="270" stroke="#475569" stroke-width="2" marker-end="url(#a)"/>
  <line x1="520" y1="270" x2="588" y2="270" stroke="#475569" stroke-width="2" marker-end="url(#a)"/>
  <line x1="715" y1="300" x2="715" y2="358" stroke="#475569" stroke-width="2" marker-end="url(#a)"/>
  <line x1="590" y1="390" x2="522" y2="390" stroke="#475569" stroke-width="2" marker-end="url(#a)"/>
  <!-- index feeds retrieve -->
  <path d="M715,150 C715,190 405,190 405,238" fill="none" stroke="#2563eb" stroke-width="2" stroke-dasharray="6 4" marker-end="url(#a)"/>
  <text x="560" y="184" text-anchor="middle" font-size="11" fill="#2563eb">reads from</text>
</svg>'''

display(SVG(RAG_DIAGRAM))

## Step 1 — Install the packages

| Package | Purpose |
|---|---|
| `azure-identity` | Log in to Azure (service principal / managed identity) |
| `azure-search-documents` | Query Azure AI Search |
| `openai` | Call the chat model |
| `python-dotenv` | Load settings from the `.env` file |

> Uses `uv`. If you don't have it, replace `!uv pip install` with `%pip install`.

In [16]:
!uv pip install --upgrade azure-identity "azure-search-documents>=12.1" openai python-dotenv

Using Python 3.12.13 environment at: /Users/najiaboo/dev/poc/env_poc
Resolved 29 packages in 485ms                                        
Checked 29 packages in 0.82ms


## Step 2 — Imports and loading the `.env` file

Secrets (keys, client secrets) must **never** be hard-coded in a notebook. We keep them in a `.env` file next to the notebook and load them as environment variables.

> `override=True` means the values in `.env` always win — so the notebook behaves the same in every fresh kernel.

Expected `.env` keys:

```
AZURE_SEARCH_ENDPOINT=https://<search-service>.search.windows.net
AZURE_SEARCH_INDEX=<index-name>
AZURE_OPENAI_ENDPOINT=<model endpoint>
AZURE_OPENAI_API_KEY=<key>
AZURE_OPENAI_MODEL=<deployment name>
AZURE_TENANT_ID=...
AZURE_CLIENT_ID=...
AZURE_CLIENT_SECRET=...
```

In [17]:
import os
import textwrap

from dotenv import load_dotenv
from azure.identity import ClientSecretCredential, ManagedIdentityCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from openai import OpenAI

loaded = load_dotenv(override=True)   # .env is the source of truth (same as test2.ipynb)
print(".env loaded:", loaded)
for k in ["AZURE_SEARCH_ENDPOINT", "AZURE_SEARCH_INDEX", "AZURE_OPENAI_ENDPOINT", "AZURE_OPENAI_API_KEY",
          "AZURE_OPENAI_MODEL", "AZURE_TENANT_ID", "AZURE_CLIENT_ID", "AZURE_CLIENT_SECRET"]:
    print(f"  {k:24} {'set' if os.getenv(k) else 'MISSING'}")

.env loaded: True
  AZURE_SEARCH_ENDPOINT    set
  AZURE_SEARCH_INDEX       set
  AZURE_OPENAI_ENDPOINT    set
  AZURE_OPENAI_API_KEY     set
  AZURE_OPENAI_MODEL       set
  AZURE_TENANT_ID          set
  AZURE_CLIENT_ID          set
  AZURE_CLIENT_SECRET      set


## Step 3 — Configuration

We read every setting into plain Python variables so students can see exactly what is used.
The three `*_FIELD` names must match the field names in **your** index (the portal wizard creates `chunk_id`, `chunk`, `title`).

In [18]:
def require(name: str) -> str:
    value = os.getenv(name, "").strip()
    if not value:
        raise ValueError(f"Missing environment variable: {name}")
    return value

SEARCH_ENDPOINT  = require("AZURE_SEARCH_ENDPOINT").rstrip("/")
SEARCH_INDEX     = require("AZURE_SEARCH_INDEX")
LLM_ENDPOINT     = require("AZURE_OPENAI_ENDPOINT").rstrip("/")
LLM_API_KEY      = require("AZURE_OPENAI_API_KEY")
LLM_MODEL        = require("AZURE_OPENAI_MODEL")

ID_FIELD      = os.getenv("RAG_ID_FIELD", "chunk_id")
CONTENT_FIELD = os.getenv("RAG_CONTENT_FIELD", "chunk")
SOURCE_FIELD  = os.getenv("RAG_SOURCE_FIELD", "title")
TOP_K         = int(os.getenv("RAG_TOP_K", "5"))

print("Search endpoint :", SEARCH_ENDPOINT)
print("Search index    :", SEARCH_INDEX)
print("LLM model       :", LLM_MODEL)
print("Fields          :", ID_FIELD, "/", CONTENT_FIELD, "/", SOURCE_FIELD)
print("Top-k           :", TOP_K)

Search endpoint : https://rag001search.search.windows.net
Search index    : rag-1795274457075
LLM model       : gpt-6-luna
Fields          : chunk_id / chunk / title
Top-k           : 5


## Step 4 — Authenticating to Azure

Azure AI Search accepts **Microsoft Entra ID** tokens (recommended) instead of API keys.

* On a laptop we use a **service principal** (`ClientSecretCredential`: tenant + client id + secret).
* When the code runs *inside* Azure (App Service, Functions, VM) we use a **managed identity** — no secrets at all.

> The identity needs the **Search Index Data Reader** role on the search service.
>
> Because this index uses **document-level access control**, the same identity also needs **Storage Blob Data Reader** on the storage container the documents came from (see Step 5b).

In [19]:
tenant_id     = os.getenv("AZURE_TENANT_ID")
client_id     = os.getenv("AZURE_CLIENT_ID")
client_secret = os.getenv("AZURE_CLIENT_SECRET")

if tenant_id and client_id and client_secret:
    credential = ClientSecretCredential(tenant_id, client_id, client_secret)
    print("Using service principal (ClientSecretCredential)")
else:
    credential = ManagedIdentityCredential(client_id=os.getenv("AZURE_MANAGED_IDENTITY_CLIENT_ID"))
    print("Using managed identity")

# Quick test: ask Entra ID for a token for Azure AI Search
token = credential.get_token("https://search.azure.com/.default")
print("Token received, expires at (epoch):", token.expires_on)

Using service principal (ClientSecretCredential)
Token received, expires at (epoch): 1795279416


## Step 5 — Connecting to Azure AI Search

A `SearchClient` talks to **one index**. Creating it does not make a network call yet.

In [20]:
search_client = SearchClient(
    endpoint=SEARCH_ENDPOINT,
    index_name=SEARCH_INDEX,
    credential=credential,
)
print("SearchClient ready for index:", SEARCH_INDEX)

SearchClient ready for index: rag-1795274457075


## Step 5b — Document-level access control (why the lab returned no answers)

This index was built with **document-level permissions**: it has an `RbacScope` field with
`permissionFilter: rbacScope`, and `permissionFilterOption: enabled`.

With that switched on, Azure AI Search only returns chunks that the identity sent in the
`x-ms-query-source-authorization` header may read in the **source storage container**.
If the header is missing, *every* search returns 0 chunks — the LLM gets no context and can only say *"I don't know"*.

So every search in this lab goes through `run_search()`, which adds that header using the
same service principal / managed identity as Step 4 (no interactive sign-in).

| `RAG_QUERY_IDENTITY` | Meaning |
|---|---|
| `app` *(default)* | Use the Step 4 identity. It needs **Storage Blob Data Reader** on the container. |
| `elevated` | Debug only: bypass the permission filter (needs elevated-read rights). |
| `none` | Send no header — only correct for indexes without a permission filter. |

In [21]:
# This index was built with document-level permissions: it has a field such as
# `RbacScope` with `permissionFilter: rbacScope` and `permissionFilterOption: enabled`.
# Azure AI Search then only returns chunks that the identity in the
# `x-ms-query-source-authorization` header is allowed to read in the source storage
# container. Without that header every search returns 0 chunks, so the LLM had no
# context and could only say "I don't know".
#
# RAG_QUERY_IDENTITY controls whose permissions are used (no interactive sign-in anywhere):
#   app       -> the same service principal / managed identity as Step 4 (default).
#                It needs "Storage Blob Data Reader" on the source storage container.
#   elevated  -> bypass the permission filter (debug only; needs elevated-read rights).
#   none      -> send nothing (only right for indexes without a permission filter).
SEARCH_SCOPE = "https://search.azure.com/.default"

permission_fields = []
try:
    with SearchIndexClient(endpoint=SEARCH_ENDPOINT, credential=credential) as index_client:
        index_def = index_client.get_index(SEARCH_INDEX)
    permission_fields = [f.name for f in index_def.fields if getattr(f, "permission_filter", None)]
    pf_option = getattr(index_def, "permission_filter_option", None)
    print("Permission filter fields:", permission_fields or "none", "| option:", pf_option)
except Exception as exc:  # reading the index definition needs extra rights - not fatal
    print(f"(could not read index definition: {str(exc)[:120]}) - assuming a permission filter")
    permission_fields = ["(unknown)"]

QUERY_IDENTITY = os.getenv("RAG_QUERY_IDENTITY", "app" if permission_fields else "none").strip().lower()
if QUERY_IDENTITY == "app":
    print("Query identity: service principal / managed identity (Step 4 credential)")
elif QUERY_IDENTITY == "elevated":
    print("Query identity: ELEVATED READ - permission filter bypassed (debug only)")
else:
    QUERY_IDENTITY = "none"
    print("Query identity: none (no x-ms-query-source-authorization header)")


def run_search(**options):
    """search_client.search() plus the permission headers this index needs."""
    if QUERY_IDENTITY == "app":
        options["query_source_authorization"] = credential.get_token(SEARCH_SCOPE).token
    elif QUERY_IDENTITY == "elevated":
        options["enable_elevated_read"] = True
    return search_client.search(**options)

Permission filter fields: ['RbacScope'] | option: SearchIndexPermissionFilterOption.ENABLED
Query identity: service principal / managed identity (Step 4 credential)


## Step 6 — Inspecting the index

**Always check this first when RAG "returns nothing".** If the document count is `0`, the indexer has not filled the index yet — the code is not the problem.

`search_text="*"` means *"match everything"*.

If the count is **greater than 0** but the sample below is empty, the index is fine but your identity is not allowed to see the documents — that is the permission filter from Step 5b.

In [22]:
print("Index:", SEARCH_INDEX)
total_chunks = search_client.get_document_count()
print("Number of chunks in index:", total_chunks)
first = next(iter(run_search(search_text="*", top=1)), None)
if first:
    print("Fields in a document:", list(first.keys()))
elif total_chunks:
    print(f"!! The index holds {total_chunks} chunks but this identity can see none of them.")
    print("   Grant the service principal / managed identity 'Storage Blob Data Reader' on the")
    print("   source storage container, or try RAG_QUERY_IDENTITY=elevated (see Step 5b).")
else:
    print("(index is empty!)")
print()

for doc in run_search(search_text="*", top=3, select=[ID_FIELD, CONTENT_FIELD, SOURCE_FIELD]):
    print("Source :", doc.get(SOURCE_FIELD))
    print("ID     :", doc.get(ID_FIELD))
    print("Chunk  :", (doc.get(CONTENT_FIELD) or "")[:200].replace("\n", " "), "...")
    print("-" * 80)

Index: rag-1795274457075


Ran into a deserialization error. Ignoring since this is failsafe deserialization
Traceback (most recent call last):
  File "/Users/najiaboo/dev/poc/env_poc/lib/python3.12/site-packages/azure/search/documents/_utils/model_base.py", line 1384, in _failsafe_deserialize
    return _deserialize(deserializer, response.json(), module, rf, format)
                                      ^^^^^^^^^^^^^^^
  File "/Users/najiaboo/dev/poc/env_poc/lib/python3.12/site-packages/azure/core/rest/_http_response_impl.py", line 331, in json
    self._json = loads(self.text())
                 ^^^^^^^^^^^^^^^^^^
  File "/Users/najiaboo/.local/share/uv/python/cpython-3.12.13-macos-aarch64-none/lib/python3.12/json/__init__.py", line 346, in loads
    return _default_decoder.decode(s)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/najiaboo/.local/share/uv/python/cpython-3.12.13-macos-aarch64-none/lib/python3.12/json/decoder.py", line 338, in decode
    obj, end = self.raw_decode(s, idx=_w(s, 0).end())
   

HttpResponseError: Operation returned an invalid status 'Forbidden'

## Step 7 — RETRIEVE: find the most relevant chunks

We send the question to Azure AI Search and get back the `top_k` best-matching chunks. Two kinds of search:

| | **Keyword (full-text)** | **Vector (semantic meaning)** |
|---|---|---|
| Matches | the exact words | the *meaning* |
| `"casual leaves"` finds `"casual leave"`? | ❌ (different word) | ✅ |
| Typos (`"cacual"`)? | ❌ | ✅ usually |

**Hybrid search** runs both and merges the results — that's what we use.
The *Import and vectorize data* wizard created a vector field (`text_vector`) **and a vectorizer**, so Azure turns our question into a vector for us (`VectorizableTextQuery`).

If the index has no vector field, the code falls back to keyword search, and if that finds nothing it retries with just the keywords of the question.

In [ ]:
import re
from azure.core.exceptions import HttpResponseError
from azure.search.documents.models import VectorizableTextQuery

VECTOR_FIELD = os.getenv("RAG_VECTOR_FIELD", "text_vector")
USE_VECTOR = True   # set to False to compare with keyword-only search

STOPWORDS = {"a", "an", "and", "are", "as", "at", "be", "by", "can", "did", "do", "does", "for", "from",
             "get", "how", "i", "in", "is", "it", "many", "much", "of", "on", "or", "that", "the", "this",
             "to", "what", "when", "where", "which", "who", "why", "with", "your"}


def keywords(question: str) -> str:
    """'How many casual leaves do employees get?' -> 'casual leaves employees'"""
    words = re.findall(r"[A-Za-z0-9_]+", question.lower())
    return " ".join(w for w in words if len(w) > 1 and w not in STOPWORDS) or question


def search_chunks(search_text: str, top_k: int, use_vector: bool) -> list[dict]:
    options = dict(search_text=search_text, top=top_k, select=[ID_FIELD, CONTENT_FIELD, SOURCE_FIELD])
    if use_vector:
        options["vector_queries"] = [
            VectorizableTextQuery(text=search_text, k_nearest_neighbors=top_k, fields=VECTOR_FIELD)
        ]
    chunks = []
    for doc in run_search(**options):
        content = (doc.get(CONTENT_FIELD) or "").strip()
        if content:
            chunks.append({
                "id": doc.get(ID_FIELD),
                "source": doc.get(SOURCE_FIELD) or "Unknown source",
                "content": content,
                "score": round(doc["@search.score"], 4),
            })
    return chunks


def retrieve(question: str, top_k: int = TOP_K) -> list[dict]:
    """Hybrid search first; keyword-only if the index has no vectorizer; then a keywords-only retry."""
    global USE_VECTOR
    if USE_VECTOR:
        try:
            return search_chunks(question, top_k, use_vector=True)
        except HttpResponseError as exc:
            print(f"(vector search not available: {exc.message[:120]} -> using keyword search)")
            USE_VECTOR = False

    chunks = search_chunks(question, top_k, use_vector=False)
    if not chunks:
        kw = keywords(question)
        print(f"(no keyword hits for the full question - retrying with: {kw!r})")
        chunks = search_chunks(kw, top_k, use_vector=False)
    return chunks


# Try it
question = "What is earned leave?"
chunks = retrieve(question)

print(f"Search mode: {'hybrid (keyword + vector)' if USE_VECTOR else 'keyword only'}")
print(f"Retrieved {len(chunks)} chunk(s) for: {question!r}\n")
for i, c in enumerate(chunks, start=1):
    print(f"[{i}] score={c['score']}  source={c['source']}")
    print(textwrap.shorten(c["content"], width=200), "\n")

## Step 8 — AUGMENT: build the prompt

We now combine three things into the messages sent to the model:

1. **System message** — the rules: *answer only from the context, cite sources*.
2. **Context** — the retrieved chunks, numbered `[1]`, `[2]`, … so the model can cite them.
3. **The question.**

This is what stops the model from "hallucinating" — it is told to use only our documents.

In [ ]:
SYSTEM_PROMPT = (
    "You are a helpful assistant that answers questions about company documents. "
    "Answer ONLY using the context provided. "
    "If the context does not contain the answer, say you don't know. "
    "Cite the sources you used with their bracketed numbers, e.g. [1]."
)


def build_messages(question: str, chunks: list[dict]) -> list[dict]:
    context = "\n\n".join(
        f"[{i}] Source: {c['source']}\n{c['content']}"
        for i, c in enumerate(chunks, start=1)
    )
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Question: {question}\n\nContext:\n{context}"},
    ]


def format_sources(chunks: list[dict]) -> str:
    """One line per source file, listing the chunk numbers [n] that came from it.
    Several chunks often come from the same PDF, so this removes the duplicates
    while keeping the [n] citations in the answer traceable."""
    by_source: dict[str, list[int]] = {}
    for i, c in enumerate(chunks, start=1):
        by_source.setdefault(c["source"], []).append(i)
    return "\n".join(
        f"  {source}  (chunks {', '.join(f'[{n}]' for n in numbers)})"
        for source, numbers in by_source.items()
    )


messages = build_messages(question, chunks)

# Look at exactly what the model will receive
print("SYSTEM:\n", messages[0]["content"], "\n")
print("USER (first 1000 chars):\n", messages[1]["content"][:1000])

## Step 9 — Connecting to the LLM

We use the `openai` Python library pointed at our Azure OpenAI / AI Foundry endpoint.

We use the `openai` Python library against Azure OpenAI's **v1 API** (`https://<resource>/openai/v1/`).

* If your endpoint already contains `/v1` we use it as-is.
* If it is a bare resource URL (or a classic `.../openai/deployments/...` URL) we build the v1 URL from the host.
  Classic URLs need an `api-version=...` on every call — without it you get **`400 Missing required query parameter: api-version`**.

In [ ]:
from urllib.parse import urlparse

_url = urlparse(LLM_ENDPOINT if "://" in LLM_ENDPOINT else "https://" + LLM_ENDPOINT)
if "/v1" in _url.path:
    LLM_BASE_URL = LLM_ENDPOINT                         # already a v1 endpoint - use as-is
else:
    LLM_BASE_URL = f"https://{_url.netloc}/openai/v1/"  # build the v1 endpoint from the host
print("LLM base URL:", LLM_BASE_URL)

llm_client = OpenAI(base_url=LLM_BASE_URL, api_key=LLM_API_KEY)

# Smoke test: a tiny request without RAG
reply = llm_client.chat.completions.create(
    model=LLM_MODEL,
    messages=[{"role": "user", "content": "Say 'hello class' in one short sentence."}],
)
print(reply.choices[0].message.content)

## Step 10 — GENERATE: get the grounded answer

We send the messages from Step 8.

> **Note on `temperature`:** with classic models (e.g. `gpt-4o`) you can pass `temperature=0` for repeatable answers.
> Reasoning models (e.g. `gpt-5`, `o3`, `o4-mini`) only accept the default and return
> **`400 'temperature' does not support 0 with this model`** — so we leave it out. The system prompt keeps answers grounded.

In [ ]:
completion = llm_client.chat.completions.create(
    model=LLM_MODEL,
    messages=messages,
)

answer = completion.choices[0].message.content
print("QUESTION:", question)
print("\nANSWER:\n", answer)
print("\nSOURCES:")
print(format_sources(chunks))

print("\nTokens used:", completion.usage.total_tokens if completion.usage else "n/a")

## Step 11 — Putting it all together: `ask()`

One function that runs the full pipeline: **Retrieve → Augment → Generate**.

In [ ]:
def ask(question: str, top_k: int = TOP_K, show_chunks: bool = False) -> str:
    # 1. Retrieve
    chunks = retrieve(question, top_k=top_k)
    if not chunks:
        return "No relevant information found in the index."

    if show_chunks:
        for i, c in enumerate(chunks, start=1):
            print(f"[{i}] score={c['score']}  {c['source']}")
        print()

    # 2. Augment
    messages = build_messages(question, chunks)

    # 3. Generate
    completion = llm_client.chat.completions.create(
        model=LLM_MODEL,
        messages=messages,
    )
    answer = completion.choices[0].message.content

    return f"{answer}\n\nSources:\n{format_sources(chunks)}"


print(ask("How many casual leaves do employees get?", show_chunks=True))

## Step 12 — Try it yourself

Change the question and run the cell again. Try:

* a question that **is** answered in the documents,
* a question that is **not** (e.g. *"Who won the World Cup?"*) — the model should say it doesn't know,
* a question with a **typo** — what happens?

In [ ]:
my_question = "What is the notice period for resignation?"
print(ask(my_question, show_chunks=True))

### Exercises

1. **Top-k:** run the same question with `top_k=1` and `top_k=10`. How does the answer change? Why?
2. **Prompt:** remove *"Answer ONLY using the context"* from `SYSTEM_PROMPT`, ask an off-topic question, and compare.
3. **Typos:** search `casual~1` with `query_type="full"` in `search_client.search(...)` (fuzzy search).
4. **Semantic ranking:** add `query_type="semantic", semantic_configuration_name="<your-config>"` and compare the ranking.
5. **Keyword vs hybrid:** set `USE_VECTOR = False` in Step 7, re-run, and ask *"How many casual leaves?"* again. Which works better, and why?

## Step 13 — Clean up

Close the network connections when finished.

In [ ]:
search_client.close()
llm_client.close()
credential.close()
print("Closed all clients.")